# CPEN355 Cat Breed Classification - Google Colab

Complete end-to-end pipeline from setup to training and inference.

Run cells in order to:
1. Setup environment
2. Install dependencies
3. Clone/setup repository
4. Train model
5. Evaluate model
6. Make predictions

In [ ]:
import os
import subprocess

# Check if running on Colab
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
print(f"Running on Colab: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    subprocess.run(['git', 'clone', 'https://github.com/yourusername/cpen355-project.git'], check=False)
    os.chdir('cpen355-project')
    print("Repository cloned")

In [ ]:
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
print("Dependencies installed")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import json
import yaml
from sklearn.metrics import confusion_matrix, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Train - CNN Model

In [ ]:
subprocess.run(['python', '-m', 'src.train', '--config', 'configs/cnn.yaml'], check=False)

## Train - ResNet50 (Baseline)

In [ ]:
subprocess.run(['python', '-m', 'src.train', '--config', 'configs/baseline.yaml'], check=False)

## Train - ResNet50 (Full Config)

In [ ]:
subprocess.run(['python', '-m', 'src.train', '--config', 'configs/resnet50.yaml'], check=False)

## Evaluate Model

In [ ]:
subprocess.run(['python', '-m', 'src.evaluate', '--config', 'configs/baseline.yaml'], check=False)

## Setup for Inference

In [ ]:
# Load config
config_path = "configs/baseline.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load label mappings
with open(Path(config['data']['processed_dir']) / 'label_to_index.json') as f:
    breed_to_idx = json.load(f)
idx_to_breed = {v: k for k, v in breed_to_idx.items()}

print(f"Loaded {len(breed_to_idx)} breeds")

In [ ]:
def load_model(config_path="configs/baseline.yaml"):
    """Load trained model from checkpoint"""
    with open(config_path) as f:
        config = yaml.safe_load(f)
    
    from src.models import CNN, ResNet50Classifier
    
    if 'cnn' in config_path.lower():
        model = CNN(num_classes=len(breed_to_idx))
    else:
        model = ResNet50Classifier(num_classes=len(breed_to_idx))
    
    ckpt_path = Path(config['paths']['checkpoint_dir']) / 'best.pt'
    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(f"Loaded model from {ckpt_path}")
    else:
        print(f"Checkpoint not found at {ckpt_path}")
        return None
    
    model.to(device)
    model.eval()
    return model

# Load model
model = load_model(config_path)

## Make Predictions

In [ ]:
def predict(img_path_or_pil, top_k=3):
    """Predict breed for an image"""
    if model is None:
        print("Model not loaded. Train model first.")
        return None
    
    if isinstance(img_path_or_pil, str) or isinstance(img_path_or_pil, Path):
        img = Image.open(img_path_or_pil).convert('RGB')
    else:
        img = img_path_or_pil
    
    from torchvision import transforms
    transform = transforms.Compose([
        transforms.Resize((config['data']['image_size'], config['data']['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    
    top_p, top_i = torch.topk(probs, min(top_k, len(breed_to_idx)))
    return [
        {'breed': idx_to_breed[i.item()], 'prob': p.item()} 
        for p, i in zip(top_p, top_i)
    ]

# Example: predict on test image
test_df = pd.read_csv(Path(config['data']['processed_dir']) / 'test.csv')
if len(test_df) > 0:
    preds = predict(test_df.iloc[0]['image_path'], top_k=3)
    print("\nTop predictions:")
    for i, p in enumerate(preds, 1):
        print(f"  {i}. {p['breed']}: {p['prob']:.2%}")
    
    # Display image
    img = Image.open(test_df.iloc[0]['image_path'])
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('\n'.join([f"{p['breed']}: {p['prob']:.1%}" for p in preds]), fontweight='bold')
    plt.tight_layout()
    plt.show()